In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [4]:
##load the trained model ,scaler pickle ,onehto
model=load_model('model.h5')

with open('onehot_encoder_gender.pkl','rb') as file:
    onehot_encoder_geo=pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)
with open('scalar.pkl','rb') as file:
    scaler=pickle.load(file)
    


In [5]:
label_encoder_gender.classes_

array(['Female', 'Male'], dtype=object)

In [6]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [7]:
input_df = pd.DataFrame([input_data])

geo_encoded = onehot_encoder_geo.transform(
    input_df[['Geography']]
).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

print(geo_encoded_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0


In [8]:
print(onehot_encoder_geo.categories_)

[array(['France', 'Germany', 'Spain'], dtype=object)]


In [9]:
input_df = pd.DataFrame([input_data])

# Drop the original Geography column
input_df = input_df.drop('Geography', axis=1)

# Add the encoded columns
input_df = pd.concat(
    [input_df.reset_index(drop=True), geo_encoded_df],
    axis=1
)

print(input_df)

   CreditScore Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600   Male   40       3    60000              2          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            50000               1.0                0.0   

   Geography_Spain  
0              0.0  


In [10]:
input_df['Gender']=label_encoder_gender.fit_transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [11]:
input_df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
print(label_encoder_gender.classes_)

['Male']


In [13]:
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516, -1.09499335,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [16]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


array([[0.06656805]], dtype=float32)

In [ ]:
prediction_proba=prediction[0][0]

In [18]:
if prediction_proba > 0.5:
    print("Customer is likely to churn")
else:
    print("Customer is not likely to churn")

Customer is not likely to churn
